# Use the engine, step by step

A first run with the ZMART Analysis engine. You make a z-stack, find its focus, then add a step that waits for a group of images. The [README](README.md) of this part is the reference.

We use the `focus` workflow that ships with the engine. It scores the sharpness of every plane in a z-stack and reports the height of the peak. No GPU, no model. It runs on a laptop.

## Before you start

- The engine installed as in [Install it](../../README.md#install-it), including the `focus` environment.
- `pip install numpy tifffile jupyter`, to make the z-stack and run this notebook.

Run the cells one at a time. The first one moves to the root of the repository, so the paths below work.

In [1]:
import os
while not os.path.exists("pyproject.toml"):
    os.chdir("..")
print(os.getcwd())

/home/user/ZMART-analysis


## 1. Make a z-stack

A noisy image, sharp at plane 4 and blurred more and more either side.

In [2]:
import numpy as np
import tifffile

def blurred(image, radius):
    out = image.astype(np.float64)
    for _ in range(radius):
        out = (np.roll(out, 1, 0) + out + np.roll(out, -1, 0)) / 3.0
        out = (np.roll(out, 1, 1) + out + np.roll(out, -1, 1)) / 3.0
    return out

sharp = np.random.default_rng(0).integers(0, 4096, size=(64, 64)).astype(np.float64)
planes = [blurred(sharp, abs(z - 4)) for z in range(9)]
tifffile.imwrite("stack.tiff", np.stack(planes).astype(np.uint16), metadata={"axes": "ZYX"})

## 2. Make an engine

`idle_timeout=None` keeps every worker alive as long as the engine lives. The default stops a worker after five minutes idle. During an acquisition, the gap between images can be longer.

In [3]:
from engine import Engine

engine = Engine(idle_timeout=None)

## 3. Register the recipe

The engine reads the recipe and finds `score_focus.py` next to it. From the `METADATA` at its top it learns the step runs in `ZMART--focus--main`. No step code has run yet.

Open the recipe. Its comment says what to submit with each job.

In [4]:
engine.register("focus", "workflows/focus/pipelines/focus.yaml")

## 4. Submit the stack

`submit` returns at once. Behind it, the engine starts a worker in `ZMART--focus--main` and hands it the stack. The first start takes a few seconds. `z_um` is the height of each plane, so the answer is a height, not a plane number.

In [5]:
z_um = [100.0 + 2.0 * z for z in range(9)]
engine.submit("focus", {"image_path": "stack.tiff", "z_um": z_um})

## 5. Read the result

In [6]:
import time

results = engine.results("focus")
while not results:
    time.sleep(0.5)
    results = engine.results("focus")

focus = results[0]["score_focus"]
print(focus["found"], focus["peak_index"], focus["peak_z_um"])

True 4.0 108.0


The peak is on plane 4, at 108 µm, as built. `focus["metrics"]` holds the sharpness curve of every metric. `results[0]["input"]` is what you submitted. `results[0]["provenance"]` says where the step ran:

In [7]:
print(results[0]["provenance"]["score_focus"]["environment"])

ZMART--focus--main


## 6. Submit several

In [8]:
for i in range(5):
    engine.submit("focus", {"image_path": "stack.tiff", "z_um": z_um})

print(engine.status("focus"))

{'pending': 5, 'running': 0, 'completed': 1, 'failed': 0, 'failures': [], 'held': {'results': 0, 'units': []}}


The worker takes them one after another, because `score_focus` has `max_workers: 1`. Wait a moment and collect:

In [9]:
time.sleep(3)
print(len(engine.results("focus")))

5


## 7. A step that waits for a group

Some analysis cannot run per image. A summary over a well needs every image of the well. Such a step has a `scope` in the recipe. It waits until a submit says the unit is complete.

Two tiny steps, in a folder of their own. They name no environment, so they run in the engine's own Python.

In [10]:
from pathlib import Path

demo = Path("demo"); (demo / "steps").mkdir(parents=True, exist_ok=True)

(demo / "steps" / "measure.py").write_text('''
def run(pipeline_data, state, **params):
    pipeline_data["measure"] = {"value": pipeline_data["input"]["value"]}
    return pipeline_data
''')

(demo / "steps" / "summarise.py").write_text('''
def run(pipeline_data, state, **params):
    values = [r["measure"]["value"] for r in pipeline_data["results"]]
    return {"n": len(values), "mean": sum(values) / len(values)}
''')

(demo / "group.yaml").write_text('''
metadata:
  functions_dir: "steps"

group_demo:
  - measure:
  - summarise:
      scope: group
''');

`measure` runs on every image. `summarise` runs once per group. It receives `pipeline_data["results"]`: the `measure` results of that group.

In [11]:
engine.register("group_demo", "demo/group.yaml")

engine.submit("group_demo", {"value": 10}, scope={"group": "A"})
engine.submit("group_demo", {"value": 20}, scope={"group": "A"})
engine.submit("group_demo", {"value": 30}, scope={"group": "A"}, complete="group")

time.sleep(2)
for r in engine.results("group_demo"):
    print(r["_phase"], r["_scope_level"], r.get("mean"))

0 None None
0 None None
0 None None
1 group 20.0


Three per-image results, then one for the group. The third submit said `complete="group"`, and that started it. The engine never guesses when a group is done. The acquisition says so.

The group result also says what it was built from:

In [12]:
print(r["lineage"]["submissions"], list(r["lineage"]["provenance"]))

[0, 1, 2] ['measure']


## 8. Shut down

In [13]:
engine.shutdown()

In a script, use the `with` form:

```python
with Engine(idle_timeout=None) as engine:
    engine.register("focus", "workflows/focus/pipelines/focus.yaml")
    ...
```

## 9. When something goes wrong

A failing step does not stop the engine. Submit a stack that does not exist:

In [14]:
with Engine() as engine:
    engine.register("focus", "workflows/focus/pipelines/focus.yaml")
    engine.submit("focus", {"image_path": "missing.tiff"})
    time.sleep(3)
    print(engine.status("focus")["failures"])

[{'scope': {}, 'step': 'score_focus', 'error': "FileNotFoundError: [Errno 2] No such file or directory: '/home/user/ZMART-analysis/missing.tiff'", 'phase': 0, 'submission_idx': 0}]


If the worker could not start at all, the message names the environment. Run that workflow's `environments/setup_env.py`.

## Clean up

Remove what this notebook made:

In [15]:
import shutil
os.remove("stack.tiff")
shutil.rmtree("demo")

## Next

- The [README](README.md) of this part: every call and argument, and the rules for scopes and workers.
- Part 2, [implement an analysis step](../2_implement_an_analysis_step/README.md).
- Part 3, [the workflows we use](../3_workflows_we_use/README.md).

---

MIT license. Thom de Hoog, Center for Microscopy and Image Analysis (ZMB), University of Zurich. thom.dehoog@zmb.uzh.ch, thomdehoog@gmail.com.